<a href="https://colab.research.google.com/github/Saptaparno20/Saptaparno20.github.io/blob/main/Quantumguard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
print("📦 Installing quantum libraries...")
!pip install -q qiskit qiskit-aer numpy pandas plotly matplotlib pylatexenc

print("✅ Installation complete!")
print("\n🚀 Now run the next cells in order")

📦 Installing quantum libraries...
✅ Installation complete!

🚀 Now run the next cells in order


In [23]:
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import circuit_drawer
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML, Markdown
import warnings
import numpy as np
warnings.filterwarnings('ignore')

print("✅ All libraries loaded successfully!")
print("\n📚 Ready to simulate quantum key distribution!")

✅ All libraries loaded successfully!

📚 Ready to simulate quantum key distribution!


In [24]:
def run_bb84_simulation(n_bits, has_unknown=False, verbose=True):
    """
    Run BB84 Protocol using actual quantum circuits

    Parameters:
    -----------
    n_bits : int - Number of qubits to transmit
    has_unknown : bool - Activate eavesdropper
    verbose : bool - Print progress messages

    Returns:
    --------
    Dictionary with all simulation results
    """

    if verbose:
        print(f"\n{'='*60}")
        print(f"🔬 STARTING BB84 QUANTUM SIMULATION")
        print(f"{'='*60}")
        print(f"📊 Qubits to transmit: {n_bits}")
        print(f"👁️  Eavesdropper (Eve): {'ACTIVE ⚠️' if has_unknown else 'Inactive ✓'}")

    # Step 1: Alice generates random bits and bases
    if verbose:
        print("\n[1/6] 👩‍💻 Alice generating random bits and bases...")
    alice_bits = np.random.randint(2, size=n_bits)
    alice_bases = np.random.randint(2, size=n_bits)  # 0=+, 1=X

    # Step 2: Bob generates random bases
    if verbose:
        print("[2/6] 👨‍💻 Bob generating random measurement bases...")
    bob_bases = np.random.randint(2, size=n_bits)

    # Step 3: Create Quantum Circuit
    if verbose:
        print("[3/6] ⚛️  Creating quantum circuit...")
    qc = QuantumCircuit(n_bits, n_bits)

    # ALICE'S ENCODING
    for i in range(n_bits):
        # Encode bit value
        if alice_bits[i] == 1:
            qc.x(i)  # X-gate: Flip |0⟩ to |1⟩

        # Encode basis choice
        if alice_bases[i] == 1:
            qc.h(i)  # Hadamard: Create superposition (rotate 45°)

        qc.barrier(i)

    # EVE'S INTERCEPTION (if active)
    unknown_bases = None
    unknown_measured = None

    if has_unknown:
        if verbose:
            print("[4/6] 🕵️  Unknown eavesdropper intercepting and measuring qubits...")
        unknown_bases = np.random.randint(2, size=n_bits)
        unknown_measured = []

        for i in range(n_bits):
            # Unknown eavesdropper measures in random basis
            if unknown_bases[i] == 1:
                qc.h(i)
            qc.measure(i, i)

            # Calculate what unknown eavesdropper measured
            if unknown_bases[i] == alice_bases[i]:
                unknown_measured.append(alice_bits[i])
            else:
                unknown_measured.append(np.random.randint(2))

            # Unknown eavesdropper re-sends (but state is disturbed!)
            qc.reset(i)
            if unknown_measured[i] == 1:
                qc.x(i)
            if unknown_bases[i] == 1:
                qc.h(i)
            qc.barrier(i)
    else:
        if verbose:
            print("[4/6] 📡 Transmitting qubits securely...")

    # BOB'S MEASUREMENT
    if verbose:
        print("[5/6] 📏 Bob measuring qubits...")
    for i in range(n_bits):
        if bob_bases[i] == 1:
            qc.h(i)
        qc.measure(i, i)

    # RUN SIMULATION
    if verbose:
        print("[6/6] 🖥️  Running quantum simulation...")
    simulator = AerSimulator()
    compiled_circuit = transpile(qc, simulator)
    job = simulator.run(compiled_circuit, shots=1, memory=True)
    result = job.result()
    measured_string = result.get_memory()[0]
    bob_measured_bits = [int(bit) for bit in measured_string[::-1]]

    # SIFTING PROCESS
    if verbose:
        print("\n🔍 Performing basis sifting...")

    sifted_key = []
    raw_key = []
    matches = 0
    errors = 0
    transmission_log = []

    for i in range(n_bits):
        match_basis = (alice_bases[i] == bob_bases[i])
        status = "Discarded"

        if match_basis:
            matches += 1
            raw_key.append(bob_measured_bits[i])
            if alice_bits[i] == bob_measured_bits[i]:
                status = "Kept ✓"
                sifted_key.append(bob_measured_bits[i])
            else:
                status = "ERROR ✗"
                errors += 1

        transmission_log.append({
            'Qubit': i+1,
            'Alice Bit': alice_bits[i],
            'Alice Basis': '✚(+)' if alice_bases[i]==0 else '❌(×)',
            'Bob Basis': '✚(+)' if bob_bases[i]==0 else '❌(×)',
            'Bob Result': bob_measured_bits[i],
            'Match': '✓' if match_basis else '✗',
            'Status': status
        })

    # Calculate metrics
    error_rate = (errors / matches * 100) if matches > 0 else 0
    is_secure = error_rate < 11
    sifting_efficiency = (matches / n_bits) * 100

    if verbose:
        print(f"✅ Sifting complete!")
        print(f"   → Matching bases: {matches}/{n_bits} ({sifting_efficiency:.1f}%)")
        print(f"   → Errors detected: {errors}")
        print(f"   → Error rate (QBER): {error_rate:.2f}%")

    # Create small circuit for visualization
    small_circuit = create_display_circuit(
        alice_bits[:5], alice_bases[:5], bob_bases[:5], has_unknown
    )

    return {
        'alice_bits': alice_bits,
        'alice_bases': alice_bases,
        'bob_bases': bob_bases,
        'bob_results': bob_measured_bits,
        'unknown_bases': unknown_bases,
        'unknown_measured': unknown_measured,
        'transmission_log': transmission_log,
        'sifted_key': sifted_key,
        'raw_key': raw_key,
        'matches': matches,
        'errors': errors,
        'error_rate': error_rate,
        'is_secure': is_secure,
        'sifting_efficiency': sifting_efficiency,
        'circuit': small_circuit
    }


def create_display_circuit(alice_bits, alice_bases, bob_bases, has_unknown):
    """Create a smaller circuit for visualization"""
    n = len(alice_bits)
    qc = QuantumCircuit(n, n)

    for i in range(n):
        if alice_bits[i] == 1:
            qc.x(i)
        if alice_bases[i] == 1:
            qc.h(i)
        qc.barrier()

    if has_unknown:
        for i in range(n):
            qc.measure(i, i)
            qc.barrier()

    for i in range(n):
        if bob_bases[i] == 1:
            qc.h(i)
        qc.measure(i, i)

    return qc

In [25]:
def display_results(results):
    """Display beautiful formatted results"""

    # Header
    display(HTML(f"""
    <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                padding: 30px; border-radius: 10px; color: white; text-align: center;
                margin-bottom: 20px;'>
        <h1 style='margin: 0; font-size: 2.5em;'>🛡️ QuantumGuard Results</h1>
        <p style='margin: 10px 0 0 0; font-size: 1.2em;'>BB84 Quantum Key Distribution</p>
    </div>
    """))

    # Metrics Dashboard
    display(HTML(f"""
    <div style='display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin-bottom: 20px;'>
        <div style='background: #f8f9fa; padding: 20px; border-radius: 8px; border-left: 4px solid #007bff;'>
            <div style='color: #6c757d; font-size: 0.9em; margin-bottom: 5px;'>🔢 QUBITS SENT</div>
            <div style='font-size: 2em; font-weight: bold; color: #212529;'>{len(results['alice_bits'])}</div>
        </div>
        <div style='background: #f8f9fa; padding: 20px; border-radius: 8px; border-left: 4px solid #28a745;'>
            <div style='color: #6c757d; font-size: 0.9em; margin-bottom: 5px;'>🔗 MATCHING BASES</div>
            <div style='font-size: 2em; font-weight: bold; color: #212529;'>{results['matches']}</div>
            <div style='font-size: 0.8em; color: #6c757d;'>{results['sifting_efficiency']:.1f}% efficiency</div>
        </div>
        <div style='background: #f8f9fa; padding: 20px; border-radius: 8px; border-left: 4px solid #ffc107;'>
            <div style='color: #6c757d; font-size: 0.9em; margin-bottom: 5px;'>🔑 FINAL KEY LENGTH</div>
            <div style='font-size: 2em; font-weight: bold; color: #212529;'>{len(results['sifted_key'])}</div>
        </div>
        <div style='background: #f8f9fa; padding: 20px; border-radius: 8px; border-left: 4px solid {"#dc3545" if results["error_rate"] > 11 else "#28a745"};'>
            <div style='color: #6c757d; font-size: 0.9em; margin-bottom: 5px;'>⚠️ ERROR RATE</div>
            <div style='font-size: 2em; font-weight: bold; color: {"#dc3545" if results["error_rate"] > 11 else "#28a745"};'>{results['error_rate']:.2f}%</div>
            <div style='font-size: 0.8em; color: #6c757d;'>Threshold: 11%</div>
        </div>
    </div>
    """))

    # Security Status
    if results['is_secure']:
        display(HTML(f"""
        <div style='background: #d4edda; border: 2px solid #28a745; border-radius: 8px;
                    padding: 20px; margin-bottom: 20px;'>
            <h2 style='color: #155724; margin: 0 0 10px 0;'>✅ SECURE CHANNEL ESTABLISHED</h2>
            <p style='color: #155724; margin: 0;'>
                Error rate ({results['error_rate']:.2f}%) is below the 11% threshold.
                The quantum channel is secure!
            </p>
            <div style='background: white; padding: 15px; border-radius: 5px; margin-top: 15px;'>
                <strong>Final Secure Key:</strong><br>
                <code style='font-size: 1.1em; color: #28a745;'>{''.join(map(str, results['sifted_key']))}</code>
            </div>
            <p style='margin: 15px 0 0 0; color: #155724;'>
                🔒 This key can now be used for AES-256 encryption!
            </p>
        </div>
        """))
    else:
        display(HTML(f"""
        <div style='background: #f8d7da; border: 2px solid #dc3545; border-radius: 8px;
                    padding: 20px; margin-bottom: 20px;'>
            <h2 style='color: #721c24; margin: 0 0 10px 0;'>🚨 CHANNEL COMPROMISED!</h2>
            <p style='color: #721c24; margin: 0;'>
                Error rate ({results['error_rate']:.2f}%) exceeds the 11% threshold!
            </p>
            <p style='margin: 15px 0 0 0; color: #721c24;'>
                {'<strong>Unknown eavesdropper was actively intercepting!</strong> This proves quantum security works - we detected the attack!' if results['unknown_bases'] is not None else 'Unexpected channel noise detected.'}
            </p>
            <p style='margin: 10px 0 0 0; color: #721c24;'>
                ⚠️ <strong>KEY DISCARDED FOR SECURITY</strong>
            </p>
        </div>
        """))

    # Transmission Log
    print("\n" + "="*80)
    print("📋 DETAILED TRANSMISSION LOG")
    print("="*80)
    df = pd.DataFrame(results['transmission_log'])
    display(df)

    print(f"\n📊 Summary Statistics:")
    print(f"   → Total transmitted: {len(results['alice_bits'])} qubits")
    print(f"   → Bases matched: {results['matches']} ({results['sifting_efficiency']:.1f}%)")
    print(f"   → Errors: {results['errors']}")
    print(f"   → QBER: {results['error_rate']:.2f}%")
    print(f"   → Final key length: {len(results['sifted_key'])} bits")


def plot_error_rate_analysis(results, has_unknown):
    """Create error rate visualization"""

    # Theoretical vs Actual
    theoretical_er = 25.0 if has_unknown else 0.0
    actual_er = results['error_rate']

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=['Theoretical', 'Your Simulation'],
        y=[theoretical_er, actual_er],
        marker_color=['lightblue', 'darkblue'],
        text=[f"{theoretical_er:.1f}%", f"{actual_er:.2f}%"],
        textposition='auto'
    ))

    fig.update_layout(
        title='Error Rate: Theory vs Practice',
        yaxis_title='Error Rate (%)',
        height=400,
        showlegend=False
    )

    fig.show()

    # Security Gauge
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=actual_er,
        title={'text': "Quantum Bit Error Rate (QBER)"},
        gauge={
            'axis': {'range': [0, 30]},
            'bar': {'color': "darkblue"},
            'steps': [
                {'range': [0, 11], 'color': "lightgreen"},
                {'range': [11, 30], 'color': "lightcoral"}
            ],
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': 11
            }
        }
    ))

    fig.update_layout(height=400)
    fig.show()


def visualize_circuit(circuit):
    """Display the quantum circuit"""
    print("\n" + "="*80)
    print("⚛️  QUANTUM CIRCUIT DIAGRAM (First 5 Qubits)")
    print("="*80)
    print("\nThis shows the actual quantum gates applied:")
    print("  • X-gate: Flips |0⟩ to |1⟩ (encodes bit value)")
    print("  • H-gate: Creates superposition (encodes basis)")
    print("  • Measure: Collapses quantum state\n")

    fig = circuit.draw('mpl', style='iqp')
    plt.show()


In [26]:
def run_interactive_demo():
    """
    Interactive demo with user input
    Run this for a full demonstration!
    """

    display(HTML("""
    <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                padding: 40px; border-radius: 15px; color: white; text-align: center;
                margin-bottom: 30px; box-shadow: 0 10px 30px rgba(0,0,0,0.3);'>
        <h1 style='margin: 0; font-size: 3em;'>🛡️ QuantumGuard</h1>
        <h2 style='margin: 10px 0; font-weight: 300;'>BB84 Quantum Key Distribution Simulator</h2>
        <p style='margin: 15px 0 0 0; font-size: 1.1em; opacity: 0.9;'>
            Privacy Preservation using Quantum Theory
        </p>
    </div>
    """))

    print("\n" + "="*80)
    print("🎮 INTERACTIVE QUANTUM SIMULATION")
    print("="*80)

    # Get user input
    print("\n📝 Configuration:")
    n_bits = int(input("   How many qubits to transmit? (10-100, recommended: 25): ") or "25")
    unknown_input = input("   Activate unknown eavesdropper? (yes/no, default: no): ").lower()
    has_unknown = unknown_input in ['yes', 'y', '1', 'true']
    show_circuit = input("   Show quantum circuit diagram? (yes/no, default: yes): ").lower()
    show_circuit = show_circuit not in ['no', 'n', '0', 'false']

    # Run simulation
    print("\n" + "="*80)
    results = run_bb84_simulation(n_bits, has_unknown, verbose=True)
    print("="*80)

    # Display results
    print("\n")
    display_results(results)

    # Show analysis
    print("\n" + "="*80)
    print("📊 STATISTICAL ANALYSIS")
    print("="*80)
    plot_error_rate_analysis(results, has_unknown)

    # Show circuit
    if show_circuit:
        visualize_circuit(results['circuit'])

    # Educational explanation
    display(Markdown(f"""
    ---

    ## 🎓 What Just Happened? (Simple Explanation)

    ### The Process:

    1. **Alice prepared {n_bits} qubits** (quantum bits = photons)
       - Each photon was polarized randomly (✚ or ❌ basis)
       - Each carried a bit value (0 or 1)

    2. **Transmission through quantum channel**
       {'- **Unknown eavesdropper intercepted!** They measured each photon with random bases' if has_unknown else '- Photons traveled securely'}
       {'- When the eavesdropper guessed wrong basis (50% chance), they DISTURBED the quantum state!' if has_unknown else ''}

    3. **Bob measured the qubits**
       - He also used random bases (✚ or ❌)
       - When his basis matched Alice's, he got the correct bit

    4. **Sifting (public basis comparison)**
       - Alice and Bob announced their bases (NOT the bits!)
       - They kept only bits where bases matched (~50%)
       - Discarded the rest

    5. **Error checking**
       - Compared some bits to calculate error rate
       - **Your result: {results['error_rate']:.2f}% error rate**
       - {'✅ Below 11% threshold → SECURE!' if results['is_secure'] else '❌ Above 11% threshold → COMPROMISED!'}

    ### The Math Behind It:

    **Without Eavesdropper:**
    - Expected error rate: ~0%
    - Only random noise from equipment

    **With Unknown Eavesdropper:**
    - Eavesdropper chooses wrong basis: 50% of the time
    - This creates errors: 50% of those times
    - **Total error rate: 0.5 × 0.5 = 25%** ← This matches theory!

    ### Why This Is Secure:

    The key insight from quantum mechanics:
    > **You cannot measure a quantum system without disturbing it!**

    - Eavesdropper MUST measure to intercept
    - Measurement COLLAPSES the superposition
    - This creates detectable errors
    - Alice & Bob know someone was listening!

    ### Real-World Applications:

    - 🏦 Banks use QKD for secure transactions
    - 🛡️ Governments use it for classified communications
    - 🛰️ China's Micius satellite does space-based QKD
    - 🔬 Research networks span 100+ km distances

    This is **NOT science fiction** - this is real technology being used TODAY!
    """))

    return results



In [27]:
def example_without_unknown():
    """Example: Secure transmission without eavesdropper"""
    print("\n" + "🔵"*40)
    print("EXAMPLE 1: SECURE TRANSMISSION (No Eavesdropper)")
    print("🔵"*40 + "\n")

    results = run_bb84_simulation(n_bits=20, has_unknown=False, verbose=True)
    print("\n")
    display_results(results)
    plot_error_rate_analysis(results, has_unknown=False)

    return results


def example_with_unknown():
    """Example: Transmission with eavesdropper attack"""
    print("\n" + "🔴"*40)
    print("EXAMPLE 2: EAVESDROPPER ATTACK (Unknown Attacker Active)")
    print("🔴"*40 + "\n")

    results = run_bb84_simulation(n_bits=20, has_unknown=True, verbose=True)
    print("\n")
    display_results(results)
    plot_error_rate_analysis(results, has_unknown=True)

    return results


def compare_scenarios():
    """Run both scenarios and compare"""
    print("\n" + "📊"*40)
    print("COMPARISON: WITH vs WITHOUT EAVESDROPPER")
    print("📊"*40 + "\n")

    print("Running secure transmission...")
    secure_results = run_bb84_simulation(30, has_unknown=False, verbose=False)

    print("Running attacked transmission...")
    attacked_results = run_bb84_simulation(30, has_unknown=True, verbose=False)

    # Comparison table
    comparison_data = {
        'Scenario': ['No Eavesdropper (Secure)', 'With Unknown Eavesdropper (Attack)'],
        'Qubits Sent': [30, 30],
        'Matching Bases': [
            secure_results['matches'],
            attacked_results['matches']
        ],
        'Errors': [
            secure_results['errors'],
            attacked_results['errors']
        ],
        'Error Rate (%)': [
            secure_results['error_rate'],
            attacked_results['error_rate']
        ],
        'Security Status': [
            '✅ SECURE' if secure_results['is_secure'] else '❌ COMPROMISED',
            '✅ SECURE' if attacked_results['is_secure'] else '❌ COMPROMISED'
        ],
        'Final Key Length': [
            len(secure_results['sifted_key']),
            len(attacked_results['sifted_key'])
        ]
    }

    df = pd.DataFrame(comparison_data)
    display(HTML("<h2>📊 Side-by-Side Comparison</h2>"))
    display(df)

    # Visual comparison
    fig = go.Figure(data=[
        go.Bar(name='No Eavesdropper', x=['Error Rate (%)'], y=[secure_results['error_rate']], marker_color='green'),
        go.Bar(name='With Eavesdropper', x=['Error Rate (%)'], y=[attacked_results['error_rate']], marker_color='red')
    ])
    fig.add_hline(y=11, line_dash="dash", line_color="orange", annotation_text="Security Threshold (11%)")
    fig.update_layout(title='Error Rate Comparison', barmode='group', height=400)
    fig.show()

    print("\n💡 Key Observations:")
    print(f"   → Without Eavesdropper: {secure_results['error_rate']:.2f}% error (near 0% as expected)")
    print(f"   → With Unknown Eavesdropper: {attacked_results['error_rate']:.2f}% error (near 25% as theory predicts)")
    print(f"   → Detection works! The eavesdropper's presence is clearly visible in the error rate.")


In [28]:
def show_educational_guide():
    """Display educational content about BB84"""

    display(Markdown("""
    # 📖 Complete BB84 Educational Guide

    ## What is Quantum Key Distribution (QKD)?

    QKD is a method to share secret encryption keys using quantum mechanics.
    Unlike classical cryptography (based on hard math problems), QKD's security
    comes from the **laws of physics** themselves!

    ---

    ## The BB84 Protocol (Step by Step)

    ### 🎯 Goal
    Alice and Bob want to share a secret key, but Eve might be listening on the channel.

    ### 📝 The Steps

    #### Step 1: Alice's Preparation
    - Alice generates random bits: `[0, 1, 1, 0, 1, ...]`
    - For each bit, she randomly chooses a polarization basis:
      - **Rectilinear ✚**: Vertical (|) or Horizontal (—)
      - **Diagonal ❌**: 45° (/) or 135° (\\)

    #### Step 2: Encoding into Photons
    - Bit 0 + Basis ✚ → Horizontal photon
    - Bit 1 + Basis ✚ → Vertical photon
    - Bit 0 + Basis ❌ → 45° photon
    - Bit 1 + Basis ❌ → 135° photon

    **In quantum circuit language:**
    ```python
    if bit == 1:
        qc.x(qubit)     # Flip to |1⟩
    if basis == ❌:
        qc.h(qubit)     # Create superposition
    ```

    #### Step 3: Transmission
    - Alice sends photons through a quantum channel (fiber optic cable)
    - **This is where Eve might intercept!**

    #### Step 4: Unknown Eavesdropper's Dilemma (if they intercept)
    - Eavesdropper catches a photon
    - They must MEASURE it to learn the bit
    - But they don't know which basis Alice used!
    - **If they guess wrong:** Measurement DESTROYS the quantum state
    - They re-send a WRONG photon to Bob

    #### Step 5: Bob's Measurement
    - Bob also randomly chooses bases for each photon
    - He measures and records the results

    #### Step 6: Basis Sifting (PUBLIC discussion)
    - Alice announces: "I used ✚, ❌, ✚, ❌, ✚..."
    - Bob announces: "I used ❌, ❌, ✚, ✚, ✚..."
    - They KEEP bits where bases matched: positions 2 and 5
    - They DISCARD the rest
    - About 50% survive (probability of random match)

    #### Step 7: Error Checking
    - They publicly compare a small subset of bits
    - Calculate error rate (QBER = Quantum Bit Error Rate)
    - **If QBER < 11%:** Channel is secure ✅
    - **If QBER ≥ 11%:** Eavesdropper detected! Abort! ❌

    #### Step 8: Privacy Amplification (if secure)
    - Compress the key to remove any partial info Eve might have
    - Uses universal hash functions
    - Final key is provably secure!

    ---

    ## 🔬 The Quantum Mechanics Behind It

    ### Key Principle 1: No-Cloning Theorem
    > You cannot make an exact copy of an unknown quantum state.

    This means Eve can't just "copy" the photon and forward the original to Bob.

    ### Key Principle 2: Measurement Disturbs
    > Measuring a quantum system in the wrong basis destroys information.

    Example:
    - Alice sends photon in state |+⟩ (diagonal basis, bit 0)
    - Eve measures in ✚ basis (wrong!)
    - Measurement collapses to |→⟩ or |↑⟩ randomly (50/50)
    - Eve re-sends this WRONG state
    - Bob measures and gets error!

    ### Key Principle 3: Heisenberg Uncertainty
    > You cannot simultaneously know both ✚ and ❌ basis values.

    These are complementary measurements - knowing one perfectly means complete
    uncertainty about the other.

    ---

    ## 📐 The Mathematics

    ### Why 25% Error Rate with Unknown Eavesdropper?

    Let's calculate:

    1. **Probability eavesdropper chooses wrong basis:** 50% (random guess)
    2. **Probability this creates an error Bob detects:** 50% (quantum mechanics)
    3. **Total error rate:** 0.5 × 0.5 = 0.25 = **25%**

    ### Why 11% Threshold?

    This comes from information theory:
    - Eavesdropper gains information from errors they don't create
    - Privacy amplification can remove their knowledge if QBER < 11%
    - Above 11%, eavesdropper knows too much - can't extract secure key

    Formula (simplified):
    ```
    Secure if: QBER < (1 - H(QBER)) / 2
    where H is binary entropy
    ```

    ---

    ## 🌍 Real-World Implementation

    ### Commercial Systems
    - **ID Quantique** (Switzerland): Commercial QKD devices
    - **Toshiba** (Japan): Quantum communication networks
    - **QuantumCTek** (China): Metropolitan QKD networks

    ### Research Milestones
    - 2007: 144 km free-space QKD achieved
    - 2016: China launches Micius quantum satellite
    - 2017: 1,200 km satellite-to-ground QKD
    - 2020: Integrated QKD with existing fiber networks

    ### Current Limitations
    - **Distance:** Photons get lost/absorbed (max ~400 km)
    - **Speed:** Slower than classical crypto
    - **Cost:** Expensive equipment required
    - **Point-to-point:** Can't route like internet traffic

    ### Future Directions
    - Quantum repeaters (extend distance)
    - Quantum internet (network multiple users)
    - Chip-scale integration (make it cheaper)
    - Post-quantum cryptography integration

    ---

    ## 🎯 For Your Hackathon Presentation

    ### Opening Statement
    "While everyone worries about quantum computers breaking encryption,
    we're using quantum mechanics to CREATE unbreakable encryption!"

    ### Key Demo Points
    1. Run without Eve → Show 0% error
    2. Activate Eve → Show ~25% error (matches theory!)
    3. Explain WHY: "You can't spy on quantum without leaving traces"
    4. Show it's REAL: "This exact protocol is used by banks TODAY"

    ### Questions You'll Get

    **Q: "Is this really quantum or just classical simulation?"**
    - A: "I'm using Qiskit - IBM's quantum SDK. Same gates (X, H) used
         on real quantum computers. The simulation uses quantum mechanics
         equations. For hackathon, I can't access actual quantum hardware,
         but the algorithm is identical."

    **Q: "Why not just use RSA encryption?"**
    - A: "RSA relies on factoring being hard. Quantum computers can break
         it (Shor's algorithm). QKD security is based on physics laws that
         even quantum computers can't break!"

    **Q: "What about the 11% threshold?"**
    - A: "It's from information theory. Below 11% error, we can use privacy
         amplification to 'squeeze out' any info Eve got. Above 11%, she
         knows too much - we abort and try again."

    **Q: "Can you scale this?"**
    - A: "Yes! China has 2,000+ km QKD network. The challenge is distance
         (photon loss). Future: quantum repeaters and satellite links."

    ---

    ## 🧪 Try It Yourself

    Run the examples below to see BB84 in action!
    """))

In [30]:
print("""
╔══════════════════════════════════════════════════════════════╗
║                                                              ║
║         🛡️  QUANTUMGUARD BB84 SIMULATOR                     ║
║         Privacy Preservation using Quantum Theory            ║
║                                                              ║
║  Ready! Choose what you want to run:                         ║
║                                                              ║
║  Option 1: run_interactive_demo()                            ║
║            → Full interactive experience with your input     ║
║                                                              ║
║  Option 2: example_without_unknown()                         ║
║            → Quick demo of secure transmission               ║
║                                                              ║
║  Option 3: example_with_unknown()                            ║
║            → Demo with eavesdropper attack                   ║
║                                                              ║
║  Option 4: compare_scenarios()                               ║
║            → Side-by-side comparison of both                 ║
║                                                              ║
║  Option 5: show_educational_guide()                          ║
║            → Learn all about BB84 protocol                   ║
║                                                              ║
║  Example usage:                                              ║
║  >>> results = run_interactive_demo()                        ║
║  >>> results = example_with_unknown()                        ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

# Uncomment the one you want to run:
# results = run_interactive_demo()
# results = example_without_unknown()
# results = example_with_unknown()
# results = compare_scenarios()
# show_educational_guide()


╔══════════════════════════════════════════════════════════════╗
║                                                              ║
║         🛡️  QUANTUMGUARD BB84 SIMULATOR                     ║
║         Privacy Preservation using Quantum Theory            ║
║                                                              ║
║  Ready! Choose what you want to run:                         ║
║                                                              ║
║  Option 1: run_interactive_demo()                            ║
║            → Full interactive experience with your input     ║
║                                                              ║
║  Option 2: example_without_unknown()                         ║
║            → Quick demo of secure transmission               ║
║                                                              ║
║  Option 3: example_with_unknown()                            ║
║            → Demo with eavesdropper attack                   ║
║                        

In [31]:
results = run_interactive_demo()


🎮 INTERACTIVE QUANTUM SIMULATION

📝 Configuration:
   How many qubits to transmit? (10-100, recommended: 25): 25
   Activate unknown eavesdropper? (yes/no, default: no): yes
   Show quantum circuit diagram? (yes/no, default: yes): no


🔬 STARTING BB84 QUANTUM SIMULATION
📊 Qubits to transmit: 25
👁️  Eavesdropper (Eve): ACTIVE ⚠️

[1/6] 👩‍💻 Alice generating random bits and bases...
[2/6] 👨‍💻 Bob generating random measurement bases...
[3/6] ⚛️  Creating quantum circuit...
[4/6] 🕵️  Unknown eavesdropper intercepting and measuring qubits...
[5/6] 📏 Bob measuring qubits...
[6/6] 🖥️  Running quantum simulation...

🔍 Performing basis sifting...
✅ Sifting complete!
   → Matching bases: 10/25 (40.0%)
   → Errors detected: 6
   → Error rate (QBER): 60.00%





📋 DETAILED TRANSMISSION LOG


,Qubit,Alice Bit,Alice Basis,Bob Basis,Bob Result,Match,Status
0,1,0,❌(×),❌(×),1,✓,ERROR ✗
1,2,0,✚(+),❌(×),1,✗,Discarded
2,3,0,✚(+),✚(+),1,✓,ERROR ✗
3,4,1,❌(×),✚(+),1,✗,Discarded
4,5,0,✚(+),✚(+),1,✓,ERROR ✗
5,6,1,❌(×),✚(+),1,✗,Discarded
6,7,1,✚(+),❌(×),1,✗,Discarded
7,8,1,❌(×),✚(+),1,✗,Discarded
8,9,1,✚(+),✚(+),1,✓,Kept ✓
9,10,0,✚(+),❌(×),1,✗,Discarded



📊 Summary Statistics:
   → Total transmitted: 25 qubits
   → Bases matched: 10 (40.0%)
   → Errors: 6
   → QBER: 60.00%
   → Final key length: 4 bits

📊 STATISTICAL ANALYSIS



    ---

    ## 🎓 What Just Happened? (Simple Explanation)

    ### The Process:

    1. **Alice prepared 25 qubits** (quantum bits = photons)
       - Each photon was polarized randomly (✚ or ❌ basis)
       - Each carried a bit value (0 or 1)

    2. **Transmission through quantum channel**
       - **Unknown eavesdropper intercepted!** They measured each photon with random bases
       - When the eavesdropper guessed wrong basis (50% chance), they DISTURBED the quantum state!

    3. **Bob measured the qubits**
       - He also used random bases (✚ or ❌)
       - When his basis matched Alice's, he got the correct bit

    4. **Sifting (public basis comparison)**
       - Alice and Bob announced their bases (NOT the bits!)
       - They kept only bits where bases matched (~50%)
       - Discarded the rest

    5. **Error checking**
       - Compared some bits to calculate error rate
       - **Your result: 60.00% error rate**
       - ❌ Above 11% threshold → COMPROMISED!

    ### The Math Behind It:

    **Without Eavesdropper:**
    - Expected error rate: ~0%
    - Only random noise from equipment

    **With Unknown Eavesdropper:**
    - Eavesdropper chooses wrong basis: 50% of the time
    - This creates errors: 50% of those times
    - **Total error rate: 0.5 × 0.5 = 25%** ← This matches theory!

    ### Why This Is Secure:

    The key insight from quantum mechanics:
    > **You cannot measure a quantum system without disturbing it!**

    - Eavesdropper MUST measure to intercept
    - Measurement COLLAPSES the superposition
    - This creates detectable errors
    - Alice & Bob know someone was listening!

    ### Real-World Applications:

    - 🏦 Banks use QKD for secure transactions
    - 🛡️ Governments use it for classified communications
    - 🛰️ China's Micius satellite does space-based QKD
    - 🔬 Research networks span 100+ km distances

    This is **NOT science fiction** - this is real technology being used TODAY!
    